In [ ]:
import cv2
import numpy as np
import qrcode


def generate_qr(data, size=200):
    """生成二维码"""
    qr = qrcode.QRCode(
        version=1,
        box_size=10,
        border=1
    )
    qr.add_data(data)
    qr.make(fit=True)

    img = qr.make_image(fill_color="black", back_color="white")
    img = np.array(img.convert("RGB"))
    img = cv2.resize(img, (size, size))

    return img


def find_plane(image, min_ratio=0.05):
    """
    找到面积大于指定比例的最小四边形平面
    """

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    edges = cv2.Canny(gray, 50, 150)

    contours, _ = cv2.findContours(
        edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    image_area = image.shape[0] * image.shape[1]
    min_area_threshold = image_area * min_ratio

    best_quad = None
    min_area = float("inf")

    for cnt in contours:

        epsilon = 0.02 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)

        if len(approx) == 4:

            area = cv2.contourArea(cnt)

            # 只考虑大于阈值的
            if area > min_area_threshold and area < min_area:

                min_area = area
                best_quad = approx

    if best_quad is None:
        return None

    return best_quad.reshape(4, 2)

def order_points(pts):
    """排序四个点"""
    rect = np.zeros((4, 2), dtype="float32")

    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    return rect


def paste_qr(image, quad, qr):
    """透视贴二维码"""
    rect = order_points(quad)

    h, w = qr.shape[:2]

    dst = np.array([
        [0, 0],
        [w, 0],
        [w, h],
        [0, h]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(dst, rect)

    warped = cv2.warpPerspective(qr, M, (image.shape[1], image.shape[0]))

    # 生成mask
    gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    mask = gray < 250

    # 让二维码不太明显
    alpha = 0.4

    result = image.copy()
    result[mask] = (
        alpha * warped[mask] +
        (1 - alpha) * result[mask]
    ).astype(np.uint8)

    return result


def main():

    img = cv2.imread("input.jpg")

    qr = generate_qr("https://example.com", 250)

    quad = find_plane(img)

    if quad is None:
        print("没有检测到平面")
        return

    result = paste_qr(img, quad, qr)

    cv2.imwrite("output.jpg", result)

    cv2.imshow("result", result)
    cv2.waitKey(0)


if __name__ == "__main__":
    main()